# Lev week 3

Цель недели: взять лучшие параметры и признаки из `lev_week2.ipynb` и полностью перебрать параметры CatBoost на датасете NASA C-MAPSS FD001.

Лучшие настройки второй недели:

- `extended_agg_window_size = 50`
- `edge_window_size = 10`
- используем все `op_setting_*` и `sensor_*`, включая `sensor_11`
- обязательные блоки: `mean`, `std`, `min`, `max`, `range`, `delta`, `slope`, `last_minus_mean`
- лучшие optional-пары: `edge_means`, `energy_acceleration`, `last_first`, `median_iqr`, `relative_changes`
- итоговый набор: `432` признака


In [ ]:
import itertools

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid


In [ ]:
columns = [
    "unit_id", "cycle", "op_setting_1", "op_setting_2", "op_setting_3",
    "sensor_1", "sensor_2", "sensor_3", "sensor_4", "sensor_5",
    "sensor_6", "sensor_7", "sensor_8", "sensor_9", "sensor_10",
    "sensor_11", "sensor_12", "sensor_13", "sensor_14", "sensor_15",
    "sensor_16", "sensor_17", "sensor_18", "sensor_19", "sensor_20", "sensor_21",
]

train_df = pd.read_csv("CMAPSSData/train_FD001.txt", sep=r"\s+", header=None, names=columns)
test_df = pd.read_csv("CMAPSSData/test_FD001.txt", sep=r"\s+", header=None, names=columns)
rul_test_df = pd.read_csv("CMAPSSData/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

target_col = "Remaining Useful Life"
train_df[target_col] = train_df["unit_id"].map(train_df.groupby("unit_id")["cycle"].max()) - train_df["cycle"]

feature_cols = [
    col for col in train_df.columns
    if col.startswith("op_setting_") or col.startswith("sensor_")
]


In [ ]:
extended_agg_window_size = 50
edge_window_size = 10
extended_agg_feature_cols = feature_cols
eps = 1e-8

mandatory_feature_blocks = [
    "mean",
    "std",
    "min",
    "max",
    "range",
    "delta",
    "slope",
    "last_minus_mean",
]

optional_feature_pairs = {
    "median_iqr": ["median", "iqr"],
    "last_first": ["last", "first"],
    "relative_changes": ["relative_delta", "relative_last_minus_mean"],
    "edge_means": ["mean_last_10", "mean_first_10"],
    "edge_change_volatility": ["mean_last_10_minus_first_10", "std_last_10"],
    "tail_quantiles": ["q10", "q90"],
    "robust_spread": ["q90_q10_range", "mad"],
    "energy_acceleration": ["rms", "slope_change"],
}

best_week2_optional_pairs = [
    "edge_means",
    "energy_acceleration",
    "last_first",
    "median_iqr",
    "relative_changes",
]

best_week2_optional_blocks = list(itertools.chain.from_iterable(
    optional_feature_pairs[pair_name]
    for pair_name in best_week2_optional_pairs
))
best_week2_blocks = mandatory_feature_blocks + best_week2_optional_blocks



In [ ]:
def calculate_slope(values):
    x = np.arange(len(values))
    x = x - x.mean()
    denominator = (x ** 2).sum()
    if denominator == 0:
        return 0
    return np.dot(values, x) / denominator


def calculate_first_edge_mean(values):
    return values[:edge_window_size].mean()


def calculate_mad(values):
    median_value = np.median(values)
    return np.median(np.abs(values - median_value))


def calculate_rms(values):
    return np.sqrt(np.mean(values ** 2))


def calculate_slope_change(values):
    split_index = len(values) // 2
    if split_index == 0 or split_index == len(values):
        return 0
    return calculate_slope(values[split_index:]) - calculate_slope(values[:split_index])


def rename_feature_block(block, suffix):
    renamed_block = block.copy()
    renamed_block.columns = [
        f"{col}_{suffix}_{extended_agg_window_size}"
        for col in renamed_block.columns
    ]
    return renamed_block


In [ ]:
def build_all_agg_window_features(source_df, include_target=False, require_full_window=False):
    sorted_df = source_df.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    min_periods = extended_agg_window_size if require_full_window else 1
    edge_min_periods = edge_window_size if require_full_window else 1

    rolling_features = sorted_df.groupby("unit_id")[extended_agg_feature_cols].rolling(
        window=extended_agg_window_size,
        min_periods=min_periods,
    )
    edge_rolling_features = sorted_df.groupby("unit_id")[extended_agg_feature_cols].rolling(
        window=edge_window_size,
        min_periods=edge_min_periods,
    )

    mean_features = rolling_features.mean().reset_index(level=0, drop=True)
    std_features = rolling_features.std(ddof=0).reset_index(level=0, drop=True).fillna(0)
    min_features = rolling_features.min().reset_index(level=0, drop=True)
    max_features = rolling_features.max().reset_index(level=0, drop=True)
    median_features = rolling_features.median().reset_index(level=0, drop=True)
    q10_features = rolling_features.quantile(0.10).reset_index(level=0, drop=True)
    q25_features = rolling_features.quantile(0.25).reset_index(level=0, drop=True)
    q75_features = rolling_features.quantile(0.75).reset_index(level=0, drop=True)
    q90_features = rolling_features.quantile(0.90).reset_index(level=0, drop=True)

    range_features = max_features - min_features
    iqr_features = q75_features - q25_features
    q90_q10_range_features = q90_features - q10_features

    first_features = sorted_df.groupby("unit_id")[extended_agg_feature_cols].shift(extended_agg_window_size - 1)
    if not require_full_window:
        first_available_values = sorted_df.groupby("unit_id")[extended_agg_feature_cols].transform("first")
        first_features = first_features.fillna(first_available_values)

    last_features = sorted_df[extended_agg_feature_cols]
    delta_features = last_features - first_features
    last_minus_mean_features = last_features - mean_features
    relative_delta_features = delta_features / (first_features.abs() + eps)
    relative_last_minus_mean_features = last_minus_mean_features / (mean_features.abs() + eps)

    mean_last_10_features = edge_rolling_features.mean().reset_index(level=0, drop=True)
    std_last_10_features = edge_rolling_features.std(ddof=0).reset_index(level=0, drop=True).fillna(0)
    mean_first_10_features = rolling_features.apply(
        calculate_first_edge_mean,
        raw=True,
    ).reset_index(level=0, drop=True)
    mean_last_10_minus_first_10_features = mean_last_10_features - mean_first_10_features

    slope_features = rolling_features.apply(
        calculate_slope,
        raw=True,
    ).reset_index(level=0, drop=True)
    mad_features = rolling_features.apply(
        calculate_mad,
        raw=True,
    ).reset_index(level=0, drop=True)
    rms_features = rolling_features.apply(
        calculate_rms,
        raw=True,
    ).reset_index(level=0, drop=True)
    slope_change_features = rolling_features.apply(
        calculate_slope_change,
        raw=True,
    ).reset_index(level=0, drop=True)

    feature_blocks = {
        "mean": mean_features,
        "std": std_features,
        "min": min_features,
        "max": max_features,
        "range": range_features,
        "delta": delta_features,
        "slope": slope_features,
        "last_minus_mean": last_minus_mean_features,
        "median": median_features,
        "iqr": iqr_features,
        "last": last_features,
        "first": first_features,
        "relative_delta": relative_delta_features,
        "relative_last_minus_mean": relative_last_minus_mean_features,
        "mean_last_10": mean_last_10_features,
        "mean_first_10": mean_first_10_features,
        "mean_last_10_minus_first_10": mean_last_10_minus_first_10_features,
        "std_last_10": std_last_10_features,
        "q10": q10_features,
        "q90": q90_features,
        "q90_q10_range": q90_q10_range_features,
        "mad": mad_features,
        "rms": rms_features,
        "slope_change": slope_change_features,
    }

    metadata_cols = ["unit_id", "cycle"]
    if include_target:
        metadata_cols.append(target_col)

    renamed_feature_blocks = {
        suffix: rename_feature_block(block, suffix)
        for suffix, block in feature_blocks.items()
    }

    result_df = pd.concat(
        [sorted_df[metadata_cols], *renamed_feature_blocks.values()],
        axis=1,
    )

    if require_full_window:
        result_df = result_df[result_df["cycle"] >= extended_agg_window_size].dropna()

    result_df = result_df.reset_index(drop=True)
    feature_block_columns = {
        suffix: list(block.columns)
        for suffix, block in renamed_feature_blocks.items()
    }

    return result_df, feature_block_columns


In [ ]:
print("Building train features...")
all_agg_window_train_df, feature_block_columns = build_all_agg_window_features(
    train_df,
    include_target=True,
    require_full_window=True,
)
print("Building test features...")
all_agg_window_test_df, _ = build_all_agg_window_features(
    test_df,
    include_target=False,
    require_full_window=False,
)

all_agg_window_test_last_rows = (
    all_agg_window_test_df
    .sort_values(["unit_id", "cycle"])
    .groupby("unit_id")
    .tail(1)
    .sort_values("unit_id")
    .reset_index(drop=True)
)

def feature_columns_for_blocks(block_names):
    selected_columns = []
    for block_name in block_names:
        selected_columns.extend(feature_block_columns[block_name])
    return selected_columns

best_week2_feature_cols = feature_columns_for_blocks(best_week2_blocks)

X_train_best_week2 = all_agg_window_train_df[best_week2_feature_cols]
y_train_best_week2 = all_agg_window_train_df[target_col].reset_index(drop=True)
X_test_best_week2 = all_agg_window_test_last_rows[best_week2_feature_cols]
y_test_best_week2 = rul_test_df["RUL"].reset_index(drop=True)

if len(X_test_best_week2) != len(y_test_best_week2):
    raise ValueError("test feature rows must match RUL target rows")

print("Features are ready")


## CatBoost parameter search

Перебираем все комбинации из `catboost_param_grid`. Во время работы каждая модель печатает номер запуска: `Training CatBoost i/N`, поэтому видно, сколько моделей уже обучилось.

Перед запуском выбери, где учить CatBoost:

- `catboost_task_type = "CPU"` - обучение на процессоре
- `catboost_task_type = "GPU"` - обучение на видеокарте


In [ ]:
catboost_task_type = "CPU"  # "CPU" или "GPU"
catboost_devices = "0"      # номер видеокарты, используется только для GPU

if catboost_task_type not in {"CPU", "GPU"}:
    raise ValueError('catboost_task_type must be "CPU" or "GPU"')

catboost_param_grid = {
    "iterations": [500, 1000, 1500],
    "learning_rate": [0.02, 0.03, 0.05],
    "depth": [4, 6, 8],
    "l2_leaf_reg": [1.0, 3.0, 5.0],
}

fixed_catboost_params = {
    "loss_function": "RMSE",
    "eval_metric": "MAE",
    "random_seed": 42,
    "verbose": 200,
    "allow_writing_files": False,
    "task_type": catboost_task_type,
}

if catboost_task_type == "GPU":
    fixed_catboost_params["devices"] = catboost_devices

catboost_grid = list(ParameterGrid(catboost_param_grid))
print(f"CatBoost task_type: {catboost_task_type}")
if catboost_task_type == "GPU":
    print(f"CatBoost GPU devices: {catboost_devices}")
print(f"Total CatBoost models to train: {len(catboost_grid)}")


In [ ]:
catboost_results = []

def evaluate_catboost_params(params, model_number, total_models):
    print(f"Training CatBoost {model_number}/{total_models} on {catboost_task_type}: {params}")
    model = CatBoostRegressor(
        **fixed_catboost_params,
        **params,
    )
    model.fit(X_train_best_week2, y_train_best_week2)

    y_pred = model.predict(X_test_best_week2)
    result = {
        **params,
        "mae": mean_absolute_error(y_test_best_week2, y_pred),
        "rmse": mean_squared_error(y_test_best_week2, y_pred) ** 0.5,
        "r2": r2_score(y_test_best_week2, y_pred),
    }
    print(
        f"Finished CatBoost {model_number}/{total_models}: "
        f"MAE={result['mae']:.6f}, RMSE={result['rmse']:.6f}, R2={result['r2']:.6f}"
    )
    return result

for model_number, params in enumerate(catboost_grid, start=1):
    catboost_results.append(
        evaluate_catboost_params(
            params=params,
            model_number=model_number,
            total_models=len(catboost_grid),
        )
    )

catboost_results_df = pd.DataFrame(catboost_results).sort_values("mae").reset_index(drop=True)
catboost_results_df


In [ ]:
best_catboost_result = catboost_results_df.iloc[0]

print("Best CatBoost parameters")
print(f"iterations: {best_catboost_result['iterations']}")
print(f"learning_rate: {best_catboost_result['learning_rate']}")
print(f"depth: {best_catboost_result['depth']}")
print(f"l2_leaf_reg: {best_catboost_result['l2_leaf_reg']}")
print(f"MAE: {best_catboost_result['mae']:.6f}")
print(f"RMSE: {best_catboost_result['rmse']:.6f}")
print(f"R2: {best_catboost_result['r2']:.6f}")
